# Notebook 05 — Backtesting on Real Data

**DS603 Group Project** · Tan Kok Yong (01426300)

This notebook performs a **rolling-window backtest** of all three VaR estimators on real CRSP returns:

1. On each day t, estimate VaR using the prior 250 trading days' losses.
2. Record whether the actual next-day loss exceeds the VaR estimate (an **exceedance**).
3. Compute the **empirical exceedance rate** and compare it to the nominal rate (1% at 99% VaR).
4. Apply a **Kupiec proportion-of-failures (POF) test** to assess whether the observed exceedance rate is statistically consistent with the nominal rate.
5. Repeat separately for calm and stressed sub-periods.



In [ ]:
# Install required packages (run once per Colab session)
!pip install -q wrds pandas numpy scipy statsmodels matplotlib seaborn pyarrow

import sys, os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from scipy import stats

from google.colab import drive
drive.mount('/content/drive')
PROJECT_ROOT = '/content/drive/MyDrive/DS603_Project'
sys.path.insert(0, PROJECT_ROOT)

from src.estimators import (
    historical_var, parametric_normal_var, parametric_t_var, fit_student_t
)

PROCESSED_DIR = os.path.join(PROJECT_ROOT, 'data', 'processed')
FIG_DIR       = os.path.join(PROJECT_ROOT, 'outputs', 'figures')
TAB_DIR       = os.path.join(PROJECT_ROOT, 'outputs', 'tables')
MDL_DIR       = os.path.join(PROJECT_ROOT, 'models')
sns.set_theme(style='whitegrid', context='notebook')

TICKERS      = ['JPM', 'AAPL', 'TSLA']
WINDOW       = 250   # rolling estimation window (trading days)
ALPHA        = 0.99  # primary confidence level

CALM_START     = pd.Timestamp('2017-01-01')
CALM_END       = pd.Timestamp('2019-12-31')
STRESSED_START = pd.Timestamp('2020-02-01')
STRESSED_END   = pd.Timestamp('2020-12-31')

In [ ]:
df = pd.read_parquet(os.path.join(PROCESSED_DIR, 'returns_clean.parquet'))
df['date'] = pd.to_datetime(df['date'])
print(f'Loaded {len(df):,} rows.')
df.head(3)

## 1. Kupiec Proportion-of-Failures (POF) test

Under a correct VaR model at level α, the probability of an exceedance on any given day is $p = 1 - \alpha$. The number of exceedances in T days follows Binomial(T, p).

The Kupiec (1995) likelihood-ratio test statistic is:

$$LR_{POF} = -2 \ln\left[\frac{p^x (1-p)^{T-x}}{(x/T)^x (1-x/T)^{T-x}}\right] \sim \chi^2_1$$

where x is the observed number of exceedances and T is the number of test days.

In [ ]:
def kupiec_pof_test(n_exceed: int, n_obs: int, alpha: float = 0.99) -> dict:
    """
    Kupiec (1995) Proportion of Failures test.

    Parameters
    ----------
    n_exceed : observed number of VaR exceedances
    n_obs    : total number of out-of-sample observations
    alpha    : VaR confidence level (1 - p_exceedance)

    Returns
    -------
    dict with observed_rate, expected_rate, lr_stat, p_value, reject_5pct
    """
    p = 1.0 - alpha  # expected exceedance probability
    x = n_exceed
    T = n_obs

    if x == 0 or x == T:
        return {
            'observed_rate': x / T,
            'expected_rate': p,
            'lr_stat': np.nan,
            'p_value': np.nan,
            'reject_5pct': None,
        }

    p_hat = x / T
    lr = -2.0 * (
        x * np.log(p / p_hat) + (T - x) * np.log((1 - p) / (1 - p_hat))
    )
    p_val = stats.chi2.sf(lr, df=1)

    return {
        'observed_rate': round(p_hat, 5),
        'expected_rate': round(p, 5),
        'lr_stat': round(lr, 4),
        'p_value': round(p_val, 4),
        'reject_5pct': bool(p_val < 0.05),
    }

# Quick sanity check
print(kupiec_pof_test(n_exceed=10, n_obs=1000, alpha=0.99))

## 2. Rolling-window backtest

In [ ]:
def rolling_backtest(losses: np.ndarray, dates: pd.Series, window: int = 250, alpha: float = 0.99) -> pd.DataFrame:
    """
    For each day t from index `window` onward:
      - Estimate VaR using losses[t-window : t]
      - Record whether losses[t] > estimated VaR (exceedance)
    """
    records = []
    n = len(losses)
    for t in range(window, n):
        train = losses[t - window : t]
        actual = losses[t]

        try:
            t_params = fit_student_t(train)
        except Exception:
            t_params = None

        var_hist = historical_var(train, alpha)
        var_norm = parametric_normal_var(train, alpha)
        var_t    = parametric_t_var(train, alpha, t_params) if t_params else np.nan

        records.append({
            'date': dates.iloc[t],
            'actual_loss': actual,
            'var_hist': var_hist,
            'var_norm': var_norm,
            'var_t':    var_t,
            'exceed_hist': int(actual > var_hist),
            'exceed_norm': int(actual > var_norm),
            'exceed_t':    int(actual > var_t) if not np.isnan(var_t) else np.nan,
        })

    return pd.DataFrame(records)


bt_results = {}
for ticker in TICKERS:
    sub = df[df['ticker'] == ticker].sort_values('date').reset_index(drop=True)
    losses = sub['loss'].values
    dates  = sub['date']
    print(f'Running backtest for {ticker} ({len(losses)} days)...')
    bt_results[ticker] = rolling_backtest(losses, dates, window=WINDOW, alpha=ALPHA)
    print(f'  Out-of-sample days: {len(bt_results[ticker])}')

print('Backtests complete.')

## 3. Exceedance timeline charts

In [ ]:
fig, axes = plt.subplots(len(TICKERS), 1, figsize=(14, 10), sharex=False)

for ax, ticker in zip(axes, TICKERS):
    bt = bt_results[ticker].copy()
    bt['date'] = pd.to_datetime(bt['date'])

    ax.plot(bt['date'], bt['actual_loss'] * 100, lw=0.5, color='steelblue', alpha=0.7, label='Actual loss')
    ax.plot(bt['date'], bt['var_hist'] * 100, lw=1.0, color='steelblue', ls='--', label='VaR Hist')
    ax.plot(bt['date'], bt['var_norm'] * 100, lw=1.0, color='crimson',   ls='--', label='VaR Norm')
    ax.plot(bt['date'], bt['var_t'] * 100,    lw=1.0, color='seagreen',  ls='--', label='VaR t')

    # Exceedances for historical method
    exc = bt[bt['exceed_hist'] == 1]
    ax.scatter(exc['date'], exc['actual_loss'] * 100, color='steelblue', s=20, zorder=5, label='Exceedance (Hist)')

    # Shade sub-periods
    ax.axvspan(CALM_START, CALM_END, alpha=0.07, color='green')
    ax.axvspan(STRESSED_START, STRESSED_END, alpha=0.07, color='red')

    ax.set_ylabel('Loss (%)')
    ax.set_title(f'{ticker} — 99% VaR rolling backtest (window={WINDOW}d)')
    ax.legend(fontsize=7, loc='upper right', ncol=3)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

plt.suptitle('Rolling-window backtest: actual losses vs. 99% VaR estimates\n(green = calm, red = stressed)', y=1.01)
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, '05_exceedance_timeline.png'), dpi=150, bbox_inches='tight')
plt.show()

## 4. Kupiec test results — full sample and sub-periods

In [ ]:
bt_rows = []
methods = ['hist', 'norm', 't']
method_labels = {'hist': 'Historical', 'norm': 'Parametric Normal', 't': 'Parametric t'}

for ticker in TICKERS:
    bt = bt_results[ticker].copy()
    bt['date'] = pd.to_datetime(bt['date'])

    subsets = {
        'full': bt,
        'calm': bt[(bt['date'] >= CALM_START) & (bt['date'] <= CALM_END)],
        'stressed': bt[(bt['date'] >= STRESSED_START) & (bt['date'] <= STRESSED_END)],
    }

    for period, sub in subsets.items():
        for m in methods:
            col = f'exceed_{m}'
            valid = sub[col].dropna()
            n_obs    = len(valid)
            n_exceed = int(valid.sum())
            if n_obs < 10:
                continue
            kupiec = kupiec_pof_test(n_exceed, n_obs, ALPHA)
            bt_rows.append({
                'Ticker': ticker,
                'Period': period,
                'Method': method_labels[m],
                'N_obs': n_obs,
                'N_exceed': n_exceed,
                'Obs rate (%)': round(kupiec['observed_rate'] * 100, 3),
                'Exp rate (%)': round(kupiec['expected_rate'] * 100, 3),
                'LR stat': kupiec['lr_stat'],
                'p-value': kupiec['p_value'],
                'Reject H0 (5%)?': 'Yes' if kupiec['reject_5pct'] else 'No',
            })

bt_table = pd.DataFrame(bt_rows)
bt_table.to_csv(os.path.join(TAB_DIR, '05_backtesting_kupiec.csv'), index=False)
bt_table

## 5. Backtesting summary and interpretation

**Reading the Kupiec test:**
- H₀: The observed exceedance rate equals the nominal rate (1% for 99% VaR).
- **Reject H₀ → the model is mis-calibrated** (either too conservative or too risky to use).
- Failure to reject → exceedances are statistically consistent with 1% frequency.

**Expected pattern under heavy tails:**
- The **Normal VaR** should be rejected in stressed periods (too many exceedances, i.e., the model understated risk).
- The **Historical and Student-t VaR** should fail to reject (well-calibrated to heavy tails).

**Stressed vs. calm sub-periods:**
- Higher exceedance rates during stressed periods are expected and confirm regime-dependent tail behaviour.
- A model that is well-calibrated in calm periods but rejected in stressed periods has failed a key robustness check.

In [ ]:
# Visual summary: observed vs. expected exceedance rates
fig, axes = plt.subplots(1, len(TICKERS), figsize=(15, 5))

for ax, ticker in zip(axes, TICKERS):
    sub = bt_table[(bt_table['Ticker'] == ticker) & (bt_table['Period'] == 'full')]
    x = np.arange(len(methods))
    obs_rates = [sub[sub['Method'] == method_labels[m]]['Obs rate (%)'].values[0]
                 if len(sub[sub['Method'] == method_labels[m]]) > 0 else 0
                 for m in methods]
    exp_rate = (1 - ALPHA) * 100

    bars = ax.bar(x, obs_rates, color=['steelblue', 'crimson', 'seagreen'], alpha=0.8,
                  label=[method_labels[m] for m in methods])
    ax.axhline(exp_rate, color='black', lw=1.5, ls='--', label=f'Nominal {exp_rate:.1f}%')
    ax.set_xticks(x)
    ax.set_xticklabels([method_labels[m] for m in methods], rotation=15, ha='right', fontsize=8)
    ax.set_ylabel('Exceedance rate (%)')
    ax.set_title(f'{ticker} (full sample)')
    ax.legend(fontsize=7)

plt.suptitle('Observed vs. nominal (1%) exceedance rate — 99% VaR backtest', fontsize=13)
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, '05_exceedance_rates.png'), dpi=150, bbox_inches='tight')
plt.show()

## 6. Decision-maker takeaway

> **Use the Student-t estimator for daily VaR and CVaR on individual equities at the 99% level.**  
> It correctly accounts for heavy tails (confirmed by the Jarque–Bera test and low fitted ν in Notebook 02), outperforms the Normal estimator on backtesting under market stress, and converges to the true value faster than historical simulation in short estimation windows (n ≤ 250).  
> The Normal estimator should be reserved for portfolios that are well-diversified enough to be near-Gaussian and is not appropriate for tail-risk measurement on individual equities.